# Core v2 — классические baseline (train → отчётные срезы)

**Задача:** обучить воспроизводимые baseline-классификаторы human vs LLM на замороженном Core и оценить по протоколу v2.

**Метрики (раздельно):** **val** (`core_val.jsonl`), **test_seen** (`core_test_non_claude.jsonl`), **test_claude_holdout** (`core_test_claude_only.jsonl`). `test_full` — только дополнительно.

**Признаки:** dense + **68 legacy HC** (`hc_*`, как в v1 `03_feature_engineering`, через NLTK) в `core_dense_features.parquet`, опционально LM NLL; TF-IDF word/char — pickle из `04_features` (fit только на train).

**Фиксация:** `random_state=42`, `class_weight='balanced'` где применимо.


In [1]:
from __future__ import annotations

import json
import pickle
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse
from scipy.sparse import csr_matrix, hstack

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.dpi"] = 120

SEED = 42
np.random.seed(SEED)


def resolve_v2_base() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if (p / "v2" / "pyproject.toml").is_file():
            return p / "v2"
        if p.name == "v2" and (p / "pyproject.toml").is_file():
            return p
    raise FileNotFoundError("Нужен v2/pyproject.toml (запуск из репозитория).")


BASE = resolve_v2_base()
FEATURE_DIR = BASE / "data" / "interim" / "features"
FIG_DIR = BASE / "outputs" / "figures" / "classical_ml"
TABLE_DIR = BASE / "outputs" / "tables" / "classical_ml"
for d in (FIG_DIR, TABLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)


BASE: /Users/askar/projects/antifraud-deepfake-detection/v2


In [2]:
DENSE_NUMERIC = [
    "char_len",
    "word_len_ws",
    "tiktoken_len",
    "digit_ratio",
    "upper_ratio",
    "punct_ratio",
    "space_ratio",
    "mean_word_len",
    "char_entropy",
    "ttr",
    "flesch_reading_ease",
    "flesch_kincaid_grade",
    "smog_index",
    "coleman_liau_index",
]

df = pd.read_parquet(FEATURE_DIR / "core_dense_features.parquet")
lm_path = FEATURE_DIR / "core_lm_distilgpt2_scores.parquet"
if lm_path.is_file():
    lm = pd.read_parquet(lm_path)
    if "core_row_id" in lm.columns and "lm_mean_nll" in lm.columns:
        df = df.drop(columns=[c for c in ("lm_mean_nll", "lm_perplexity") if c in df.columns], errors="ignore")
        df = df.merge(lm[["core_row_id", "lm_mean_nll"]], on="core_row_id", how="left")
        print("Merged LM scores from", lm_path)
else:
    print("No LM cache — run 04_features with lm_scoring optional extra if needed")

USE_LM = "lm_mean_nll" in df.columns and df["lm_mean_nll"].notna().any()
LEGACY_HC_COLS = sorted([c for c in df.columns if str(c).startswith("hc_")])
num_cols = list(DENSE_NUMERIC) + LEGACY_HC_COLS + (["lm_mean_nll"] if USE_LM else [])
print("num_cols: dense", len(DENSE_NUMERIC), "hc", len(LEGACY_HC_COLS), "lm", int(USE_LM), "total", len(num_cols))

train_m = df["_load_split"] == "train"
val_m = df["_load_split"] == "val"
ts_m = df["_load_split"] == "test_non_claude"
tcl_m = df["_load_split"] == "test_claude_only"

split_masks = {
    "train": train_m,
    "val": val_m,
    "test_seen": ts_m,
    "test_claude_holdout": tcl_m,
}

for k, m in split_masks.items():
    print(k, m.sum(), "label balance", df.loc[m, "label"].value_counts().to_dict())

with (FEATURE_DIR / "core_tfidf_word_vectorizer.pkl").open("rb") as f:
    vec_w = pickle.load(f)
with (FEATURE_DIR / "core_tfidf_char_vectorizer.pkl").open("rb") as f:
    vec_c = pickle.load(f)

def tf_stack(sub: pd.DataFrame):
    t = sub["text"].astype(str)
    return vec_w.transform(t), vec_c.transform(t)

Xw = {}
Xc = {}
a0, b0 = tf_stack(df.loc[train_m])
nw, nc = a0.shape[1], b0.shape[1]
for name, m in split_masks.items():
    sub = df.loc[m]
    if len(sub) == 0:
        Xw[name] = sparse.csr_matrix((0, nw))
        Xc[name] = sparse.csr_matrix((0, nc))
    else:
        a, b = tf_stack(sub)
        Xw[name] = a
        Xc[name] = b

print("TF-IDF word dim", nw, "char", nc)


Merged LM scores from /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/features/core_lm_distilgpt2_scores.parquet
num_cols: dense 14 hc 68 lm 1 total 83
train 13554 label balance {0: 6777, 1: 6777}
val 2706 label balance {0: 1353, 1: 1353}
test_seen 1816 label balance {0: 908, 1: 908}
test_claude_holdout 2059 label balance {1: 2059}
TF-IDF word dim 8000 char 4000


In [3]:
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.preprocessing import StandardScaler

imp = SimpleImputer(strategy="median")
sc = StandardScaler()
imp.fit(df.loc[train_m, num_cols].astype(float))
sc.fit(imp.transform(df.loc[train_m, num_cols].astype(float)))

Xd = {}
for name, m in split_masks.items():
    Xd[name] = sc.transform(imp.transform(df.loc[m, num_cols].astype(float)))

y = {k: df.loc[m, "label"].astype(int).values for k, m in split_masks.items()}

X_full = {
    k: hstack([csr_matrix(Xd[k]), Xw[k], Xc[k]], format="csr")
    for k in split_masks
}

print("X_full train", X_full["train"].shape)


X_full train (13554, 12083)


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False


def eval_split(clf, X, y_true, need_proba=True):
    if need_proba and hasattr(clf, "predict_proba"):
        p = clf.predict_proba(X)[:, 1]
    else:
        p = clf.decision_function(X)
        p = (p - p.min()) / (p.max() - p.min() + 1e-12)
    pred = (p >= 0.5).astype(int)
    out = {
        "accuracy": float(accuracy_score(y_true, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred)),
        "roc_auc": float("nan"),
        "avg_precision": float("nan"),
        "f1": float(f1_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "scores": p,
        "pred": pred,
    }
    if np.unique(y_true).size >= 2:
        out["roc_auc"] = float(roc_auc_score(y_true, p))
        out["avg_precision"] = float(average_precision_score(y_true, p))
    else:
        # В Core v2 в test_claude_only сейчас только label=1 (LLM) — нет пары human для ROC/AP
        out["llm_predicted_rate"] = float(np.mean(pred == 1))
    return out


def plot_curves(model_slug, clf, X_dict, y_dict, splits_plot):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, spl in zip(axes, splits_plot):
        yt = y_dict[spl]
        r = eval_split(clf, X_dict[spl], yt)
        if np.unique(yt).size >= 2:
            RocCurveDisplay.from_predictions(yt, r["scores"], ax=ax, name=spl)
        else:
            ax.text(0.5, 0.5, "ROC N/A\n(один класс в y)", ha="center", va="center")
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
        ax.set_title(f"ROC — {spl}")
    fig.suptitle(f"{model_slug} — ROC")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{model_slug}_roc.png", dpi=150)
    plt.close(fig)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, spl in zip(axes, splits_plot):
        yt = y_dict[spl]
        r = eval_split(clf, X_dict[spl], yt)
        if np.unique(yt).size >= 2:
            PrecisionRecallDisplay.from_predictions(yt, r["scores"], ax=ax, name=spl)
        else:
            ax.text(0.5, 0.5, "PR N/A\n(один класс в y)", ha="center", va="center")
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
        ax.set_title(f"PR — {spl}")
    fig.suptitle(f"{model_slug} — PR")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{model_slug}_pr.png", dpi=150)
    plt.close(fig)

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
    for ax, spl in zip(axes, splits_plot):
        r = eval_split(clf, X_dict[spl], y_dict[spl])
        cm = confusion_matrix(y_dict[spl], r["pred"])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
        ax.set_xlabel("predicted")
        ax.set_ylabel("true")
        ax.set_title(spl)
    fig.suptitle(f"{model_slug} — confusion")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{model_slug}_confusion.png", dpi=150)
    plt.close(fig)


EVAL_SPLITS = ["val", "test_seen", "test_claude_holdout"]
rows = []


In [5]:
# 1) LogisticRegression: dense + TF-IDF (главный sparse baseline)
lr_full = LogisticRegression(
    max_iter=300,
    class_weight="balanced",
    solver="saga",
    n_jobs=-1,
    random_state=SEED,
)
lr_full.fit(X_full["train"], y["train"])

for spl in EVAL_SPLITS:
    r = eval_split(lr_full, X_full[spl], y[spl])
    row = {k: r[k] for k in r if k not in ("scores", "pred")}
    rows.append({"model": "lr_dense_tfidf", "split": spl, **row})

plot_curves("lr_dense_tfidf", lr_full, X_full, y, EVAL_SPLITS)
print("lr_dense_tfidf done")


/Users/askar/projects/antifraud-deepfake-detection/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


lr_dense_tfidf done


In [6]:
# 2) LogisticRegression: только dense (+LM)
lr_d = LogisticRegression(
    max_iter=200,
    class_weight="balanced",
    solver="lbfgs",
    random_state=SEED,
)
lr_d.fit(Xd["train"], y["train"])

for spl in EVAL_SPLITS:
    r = eval_split(lr_d, Xd[spl], y[spl])
    rows.append({"model": "lr_dense_only", "split": spl, **{k: r[k] for k in r if k not in ("scores", "pred")}})

plot_curves("lr_dense_only", lr_d, Xd, y, EVAL_SPLITS)
print("lr_dense_only done")


lr_dense_only done


In [7]:
# 3) RandomForest на dense (+LM)
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=24,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=SEED,
    n_jobs=-1,
)
rf.fit(Xd["train"], y["train"])

for spl in EVAL_SPLITS:
    r = eval_split(rf, Xd[spl], y[spl])
    rows.append({"model": "rf_dense", "split": spl, **{k: r[k] for k in r if k not in ("scores", "pred")}})

plot_curves("rf_dense", rf, Xd, y, EVAL_SPLITS)
print("rf_dense done")


rf_dense done


In [8]:
# 4) XGBoost на dense (если доступен)
xgb_clf = None
if HAS_XGB:
    xgb_clf = xgb.XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.08,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=-1,
        eval_metric="logloss",
    )
    xgb_clf.fit(Xd["train"], y["train"], verbose=False)
    for spl in EVAL_SPLITS:
        r = eval_split(xgb_clf, Xd[spl], y[spl])
        rows.append({"model": "xgb_dense", "split": spl, **{k: r[k] for k in r if k not in ("scores", "pred")}})
    plot_curves("xgb_dense", xgb_clf, Xd, y, EVAL_SPLITS)
    print("xgb_dense done")
else:
    print("xgboost not available — skip")


xgb_dense done


In [9]:
# Сводная таблица + test_full (агрегат): объединение test_seen ∪ test_claude_holdout по строкам df
m_tf = df["_load_split"].isin(["test_non_claude", "test_claude_only"])
if m_tf.sum() > 0:
    imp_tf = SimpleImputer(strategy="median")
    sc_tf = StandardScaler()
    imp_tf.fit(df.loc[train_m, num_cols].astype(float))
    sc_tf.fit(imp_tf.transform(df.loc[train_m, num_cols].astype(float)))
    Xd_tf = sc_tf.transform(imp_tf.transform(df.loc[m_tf, num_cols].astype(float)))
    tw, tc = tf_stack(df.loc[m_tf])
    X_tf = hstack([csr_matrix(Xd_tf), tw, tc], format="csr")
    y_tf = df.loc[m_tf, "label"].astype(int).values
    for model_name, clf in [
        ("lr_dense_tfidf", lr_full),
        ("lr_dense_only", lr_d),
        ("rf_dense", rf),
    ] + ([("xgb_dense", xgb_clf)] if HAS_XGB and xgb_clf is not None else []):
        if model_name == "lr_dense_tfidf":
            r = eval_split(clf, X_tf, y_tf)
        else:
            r = eval_split(clf, Xd_tf, y_tf)
        rows.append({"model": model_name, "split": "test_full_aggregate", **{k: r[k] for k in r if k not in ("scores", "pred")}})

metrics_df = pd.DataFrame(rows)
metrics_path = TABLE_DIR / "baseline_metrics_by_split.csv"
metrics_df.to_csv(metrics_path, index=False)
print(metrics_df.to_string(index=False))
print("Wrote", metrics_path)


         model               split  accuracy  balanced_accuracy  roc_auc  avg_precision       f1  precision   recall  llm_predicted_rate
lr_dense_tfidf                 val  0.918699           0.918699 0.963765       0.939767 0.920405   0.901488 0.940133                 NaN
lr_dense_tfidf           test_seen  0.916850           0.916850 0.972045       0.969358 0.919035   0.895507 0.943833                 NaN
lr_dense_tfidf test_claude_holdout  0.631860           0.631860      NaN            NaN 0.774405   1.000000 0.631860            0.631860
 lr_dense_only                 val  0.906504           0.906504 0.955439       0.931734 0.908167   0.892297 0.924612                 NaN
 lr_dense_only           test_seen  0.905286           0.905286 0.967425       0.964252 0.907127   0.889831 0.925110                 NaN
 lr_dense_only test_claude_holdout  0.470617           0.470617      NaN            NaN 0.640026   1.000000 0.470617            0.470617
      rf_dense                 val  0.957

## Диагностика по `scenario_family` (test_seen)

Ориентировочные метрики **только** для среза `test_seen` — показывают неоднородность сценариев.


In [10]:
rows_sf = []
sub = df.loc[ts_m].copy()
if len(sub) > 0:
    imp2 = SimpleImputer(strategy="median")
    sc2 = StandardScaler()
    imp2.fit(df.loc[train_m, num_cols].astype(float))
    sc2.fit(imp2.transform(df.loc[train_m, num_cols].astype(float)))
    for fam in sorted(sub["scenario_family"].astype(str).unique()):
        mloc = sub["scenario_family"].astype(str) == fam
        if mloc.sum() < 30:
            continue
        idx = sub.index[mloc]
        Xloc = hstack(
            [
                csr_matrix(
                    sc2.transform(imp2.transform(df.loc[idx, num_cols].astype(float)))
                ),
                vec_w.transform(df.loc[idx, "text"].astype(str)),
                vec_c.transform(df.loc[idx, "text"].astype(str)),
            ],
            format="csr",
        )
        ylab = df.loc[idx, "label"].astype(int).values
        if np.unique(ylab).size < 2:
            continue
        r = eval_split(lr_full, Xloc, ylab)
        rows_sf.append({"scenario_family": fam, "n": int(mloc.sum()), **{k: r[k] for k in r if k not in ("scores", "pred")}})
    if rows_sf:
        sf_df = pd.DataFrame(rows_sf).sort_values("roc_auc")
        sf_path = TABLE_DIR / "baseline_by_scenario_family_test_seen.csv"
        sf_df.to_csv(sf_path, index=False)
        print(sf_df.to_string(index=False))
        print("Wrote", sf_path)
    else:
        print("No per-family rows with both classes")
else:
    print("No test_seen rows")


       scenario_family   n  accuracy  balanced_accuracy  roc_auc  avg_precision       f1  precision   recall
          financial_qa 164  0.762195           0.762195 0.830607       0.852616 0.763636   0.759036 0.768293
        legitimate_sms 160  0.737500           0.737500 0.866094       0.877191 0.761364   0.697917 0.837500
   fraud_sms_deceptive 172  0.825581           0.825581 0.928880       0.923809 0.827586   0.818182 0.837209
      legitimate_email 400  0.962500           0.962500 0.991375       0.987378 0.963145   0.946860 0.980000
        phishing_email 600  0.961667           0.961667 0.997700       0.997935 0.962963   0.931464 0.996667
advance_fee_scam_email 320  0.993750           0.993750 0.999961       0.999961 0.993789   0.987654 1.000000
Wrote /Users/askar/projects/antifraud-deepfake-detection/v2/outputs/tables/classical_ml/baseline_by_scenario_family_test_seen.csv


## Где лежат артефакты

- Таблицы: `v2/outputs/tables/classical_ml/`
- Фигуры: `v2/outputs/figures/classical_ml/` (`*_roc.png`, `*_pr.png`, `*_confusion.png`)
